# Stage 1: What is RAG? Load documents and split them into chunks

This notebook introduces the **RAG pipeline** and practices two building blocks:

1. **Loading** text documents from our knowledge base
2. **Chunking** that text into smaller pieces

We use plain Python only (no LangChain, embeddings, vector DB, or LLM yet).

## The RAG idea (big picture)

**RAG** means **Retrieve → Augment → Generate**:

1. **Prepare documents** — FAQs, policies, product notes (our knowledge base)
2. **Split into chunks** — smaller pieces are easier to search and fit into a prompt
3. **Embed and store** *(later stages)* — turn chunks into vectors; save them in a vector database
4. **Retrieve** *(later)* — for a user question, find the most relevant chunks
5. **Generate** *(later)* — send those chunks + the question to a local LLM (Ollama) and answer from that context

Why RAG? A plain LLM may guess. RAG grounds answers in **your** documents. If nothing relevant is found, the bot should say it does not know.

```
Documents → Chunks → (Embeddings → Vector DB) → Retrieve → Prompt + LLM → Answer
                 ▲
           this notebook
```

## Setup: locate the knowledge base

Sample files live in `data/knowledge_base/` (Harbor & Co. policies plus a fictional product FAQ).

In [ ]:
from pathlib import Path

# Notebooks run from the notebooks/ folder or the project root — support both.
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data" / "knowledge_base").is_dir():
    pass
elif (PROJECT_ROOT.parent / "data" / "knowledge_base").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError(
        "Could not find data/knowledge_base/. Open the notebook from the project root or notebooks/."
    )

KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"
print(f"Project root: {PROJECT_ROOT}")
print(f"Knowledge base: {KB_DIR}")
print("Files:")
for path in sorted(KB_DIR.glob("*.txt")):
    print(f"  - {path.name} ({path.stat().st_size} bytes)")

## Load documents

Each `.txt` file becomes one document: a small dictionary with `source` (filename) and `text` (full content).

In [ ]:
def load_text_documents(folder: Path) -> list[dict]:
    """Load every .txt file in folder into a list of {source, text} dicts."""
    documents = []
    for path in sorted(folder.glob("*.txt")):
        text = path.read_text(encoding="utf-8").strip()
        documents.append({"source": path.name, "text": text})
    return documents


docs = load_text_documents(KB_DIR)
print(f"Loaded {len(docs)} documents\n")

for doc in docs:
    preview = doc["text"][:120].replace("\n", " ")
    print(f"{doc['source']}")
    print(f"  chars: {len(doc['text'])}")
    print(f"  preview: {preview}...\n")

## Why chunk?

- A full policy file may be longer than we want to stuff into every LLM prompt.
- Retrieval works better on **focused** pieces (e.g. “refund timing” vs the whole refund policy).
- Later we will embed each chunk and search by similarity to the user question.

Two simple strategies below (still plain Python):

1. **By paragraph** — split on blank lines
2. **By character window** — fixed size with optional overlap

### Strategy A: paragraph chunks

In [ ]:
def chunk_by_paragraphs(text: str) -> list[str]:
    """Split text on blank lines; drop empty pieces."""
    parts = [p.strip() for p in text.split("\n\n")]
    return [p for p in parts if p]


sample = next(d for d in docs if d["source"] == "refund-policy.txt")
para_chunks = chunk_by_paragraphs(sample["text"])

print(f"Document: {sample['source']}")
print(f"Paragraph chunks: {len(para_chunks)}\n")
for i, chunk in enumerate(para_chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

### Strategy B: fixed-size character chunks with overlap

**Overlap** keeps sentences near a boundary from being split awkwardly across two chunks.

In [ ]:
def chunk_by_characters(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """Sliding window over characters. overlap must be < chunk_size."""
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks


char_chunks = chunk_by_characters(sample["text"], chunk_size=180, overlap=40)

print(f"Document: {sample['source']}")
print(f"Character chunks: {len(char_chunks)} (size=180, overlap=40)\n")
for i, chunk in enumerate(char_chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(repr(chunk))
    print()

## Chunk the full knowledge base

We attach metadata (`source`, `chunk_id`) so later stages can show *which* document a retrieved piece came from.

In [ ]:
def build_chunk_records(documents: list[dict]) -> list[dict]:
    """Create paragraph chunks for every document with simple metadata."""
    records = []
    for doc in documents:
        for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
            records.append(
                {
                    "source": doc["source"],
                    "chunk_id": i,
                    "text": chunk,
                    "char_count": len(chunk),
                }
            )
    return records


all_chunks = build_chunk_records(docs)

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chars/chunk: {sum(c['char_count'] for c in all_chunks) / len(all_chunks):.0f}\n")

from collections import Counter

by_source = Counter(c["source"] for c in all_chunks)
for name, count in sorted(by_source.items()):
    print(f"  {name}: {count} chunks")

print("\nExample chunk record:")
example = all_chunks[0]
for key, value in example.items():
    print(f"  {key}: {value!r}")

## What you should notice

- Loading is just reading files into structured objects.
- Chunking trades off **size** (too big → noisy retrieval; too small → missing context) and **boundaries** (paragraphs often read better than mid-sentence cuts).
- Metadata (`source`, `chunk_id`) will matter when we show citations or debug retrieval.

## Next stage (not built yet)

**Stage 2:** turn these chunks into **embeddings** (numeric vectors) with a local embedding model and compare similarity between a question and chunks — still before wiring Chroma or Ollama chat.